# Домашнее задание 5

Я объединила данные о визитах и покупках по `user_id`. В `funnel.csv` записала только визиты пользователей, которые есть в файле покупок, и добавила категорию покупки.

## Подготовка файлов

Я использую `purchase_log.txt` и `visit_log.csv` из выданного архива с материалами задания. Перед запуском размещаю их в одной папке с notebook. Для решения использую Python 3 и стандартные модули `json` и `csv`.

In [1]:
import json
import csv

## 1. Словарь покупок

Я читаю JSON-записи построчно и сохраняю категории в словарь по `user_id`. Пропускаю первую запись с названиями полей. Для повторяющегося `user_id` сохраняю категорию из последней записи в файле. Так для каждого визита я получаю одну категорию.

In [2]:
purchases = {}

with open('purchase_log.txt', encoding='utf-8') as purchase_file:
    for line in purchase_file:
        if not line.strip():
            continue
        purchase = json.loads(line)
        if purchase['user_id'] == 'user_id' and purchase['category'] == 'category':
            continue
        purchases[purchase['user_id']] = purchase['category']

print('Пользователей с покупками:', len(purchases))

Пользователей с покупками: 99517


## 2. Построчная обработка визитов

Я перебираю строки `visit_log.csv` через `csv.reader`, не загружая весь файл в память. Если нахожу пользователя в словаре покупок, сразу записываю визит и категорию в `funnel.csv`. Сохраняю порядок визитов и повторные визиты. Исходный файл открываю только для чтения.

In [3]:
visit_count = 0
funnel_count = 0

with open('visit_log.csv', encoding='utf-8', newline='') as visit_file, \
        open('funnel.csv', 'w', encoding='utf-8', newline='') as funnel_file:
    reader = csv.reader(visit_file)
    writer = csv.writer(funnel_file, lineterminator='\n')
    next(reader)  # Пропускаю заголовок исходного файла.
    writer.writerow(['user_id', 'source', 'category'])

    for user_id, source in reader:
        visit_count += 1
        if user_id in purchases:
            writer.writerow([user_id, source, purchases[user_id]])
            funnel_count += 1

print('Обработано визитов:', visit_count)
print('Записано в funnel.csv:', funnel_count)

Обработано визитов: 199999
Записано в funnel.csv: 13798


## 3. Первые три строки

Я вывожу заголовок и первые две строки результата и сравниваю их с примером задания.

In [4]:
with open('funnel.csv', encoding='utf-8') as funnel_file:
    for _ in range(3):
        print(next(funnel_file).rstrip('\n'))

user_id,source,category
1840e0b9d4,other,Продукты
4e4f90fcfb,context,Электроника


## 4. Самопроверка

Я проверяю заголовок, первые строки, количество записей и категории. Повторно сопоставляю результат с исходными визитами построчно, чтобы проверить отсутствие пропусков и сохранение порядка.

In [5]:
with open('funnel.csv', encoding='utf-8', newline='') as funnel_file:
    reader = csv.reader(funnel_file)
    assert next(reader) == ['user_id', 'source', 'category']
    assert next(reader) == ['1840e0b9d4', 'other', 'Продукты']
    assert next(reader) == ['4e4f90fcfb', 'context', 'Электроника']

checked_count = 0
with open('visit_log.csv', encoding='utf-8', newline='') as visit_file, \
        open('funnel.csv', encoding='utf-8', newline='') as funnel_file:
    visits = csv.reader(visit_file)
    result = csv.reader(funnel_file)
    next(visits)
    next(result)
    for user_id, source in visits:
        if user_id in purchases:
            assert next(result) == [user_id, source, purchases[user_id]]
            checked_count += 1
    assert next(result, None) is None

assert checked_count == funnel_count
print('Все проверки пройдены.')

Все проверки пройдены.
